# درس 10: گام تصادفی، حرکت براونی و GBM

متن کامل درس: [https://quantacademy.ir/courses/quant-math/random-walk-brownian-gbm/](https://quantacademy.ir/courses/quant-math/random-walk-brownian-gbm/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. گام تصادفی و قانون ریشه زمان

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import norm

rng = np.random.default_rng(10)
n_paths, n_steps, sigma_step = 20000, 256, 0.02
steps = rng.normal(0, sigma_step, (n_paths, n_steps))
walk = steps.cumsum(axis=1)
print("t     std of walk   sigma*sqrt(t)")
for t in (1, 4, 16, 64, 256):
    print(f"{t:<5d} {walk[:, t - 1].std():11.4f} {sigma_step * np.sqrt(t):14.4f}")

## ۲. حرکت براونی

## ۳. چرا نوسان کریپتو با √۳۶۵ سالانه می‌شود؟

In [ ]:
URL = "https://raw.githubusercontent.com/quantacademyir/quant-math/main/data/btc_daily_5y.csv"
close = pd.read_csv(URL, parse_dates=["date"], index_col="date")["close"]
ret = np.log(close).diff().dropna()
daily_std = ret.std()
print(f"daily std {daily_std:.4f}   x sqrt(365) = {daily_std * np.sqrt(365):.3f}")
for h in (1, 7, 30):
    r_h = np.log(close).diff(h).dropna().iloc[::h]
    print(f"{h:>2d}-day returns: n={len(r_h):4d}  std {r_h.std():.4f}  daily std x sqrt({h}) = {daily_std * np.sqrt(h):.4f}")

## ۴. حرکت براونی هندسی (GBM)

In [ ]:
S0, mu, sigma, T = 100.0, 0.30, 0.65, 1.0
rng = np.random.default_rng(11)
W_T = rng.normal(0, np.sqrt(T), 400_000)
S_T = S0 * np.exp((mu - sigma ** 2 / 2) * T + sigma * W_T)
print(f"mean   simulated {S_T.mean():7.2f}   theory S0*exp(mu*T) = {S0 * np.exp(mu * T):7.2f}")
print(f"median simulated {np.median(S_T):7.2f}   theory S0*exp((mu-sigma^2/2)*T) = {S0 * np.exp((mu - sigma ** 2 / 2) * T):7.2f}")
p_down = (S_T < S0).mean()
theory = norm.cdf(-(mu - sigma ** 2 / 2) * np.sqrt(T) / sigma)
print(f"P(S_T < S0) simulated {p_down:.3f}   theory {theory:.3f}")
print(f"5%..95% of S_T: {np.percentile(S_T, 5):.1f} .. {np.percentile(S_T, 95):.1f}")

In [ ]:
wrong = S0 * np.exp(mu * T + sigma * W_T)
print(f"without the -sigma^2/2 term: mean {wrong.mean():.1f}  (theory would be {S0 * np.exp(mu * T + sigma ** 2 * T / 2):.1f})")

## ۵. از داده چه چیزی می‌شود برآورد کرد؟

In [ ]:
n = len(ret)
years = n / 365
mean_daily = ret.mean()
mu_log = mean_daily * 365
sigma_ann = daily_std * np.sqrt(365)
se_mu = sigma_ann / np.sqrt(years)
se_sigma = sigma_ann / np.sqrt(2 * n)
print(f"years {years:.2f}")
print(f"drift of log price per year  {mu_log:+.3f}  +- 2 se: {2 * se_mu:.3f}")
print(f"mu = drift + sigma^2/2       {mu_log + sigma_ann ** 2 / 2:+.3f}")
print(f"sigma per year               {sigma_ann:.3f}  +- 2 se: {2 * se_sigma:.3f}")

## ۶. GBM مدل است، نه واقعیت

In [ ]:
z = (ret - ret.mean()) / ret.std()
big = (z.abs() > 4).sum()
print(f"days beyond 4 sigma: {big}   expected under normal: {n * 2 * norm.sf(4):.2f}")
print(f"largest daily move: {z.abs().max():.1f} sigma")